# Figures S20-S21

Evaluates the Slow-recovery probability model with spatially blocked diagnostics.


In [1]:
from pathlib import Path
import json
import warnings

import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import GroupKFold
from shapely.geometry import LineString

warnings.filterwarnings('ignore', category=RuntimeWarning)

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'outputs' / 'RECON_MAIN_2011_2023').exists()
)
MAIN_NOTEBOOK = ROOT / 'notebooks' / '4_regression.ipynb'
REGRESSION_DIR = ROOT / 'outputs' / 'RECON_MAIN_2011_2023' / 'metrics' / 'regression'
FIG_DIR = ROOT / 'outputs' / 'figures' / 'FigS20'
FIG21_DIR = ROOT / 'outputs' / 'figures' / 'FigS21'
SOURCE_DIR = FIG_DIR / 'source_data'
SOURCE21_DIR = FIG21_DIR / 'source_data'
FIG_DIR.mkdir(parents=True, exist_ok=True)
FIG21_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
SOURCE21_DIR.mkdir(parents=True, exist_ok=True)

FIG_PATH = FIG_DIR / 'FigS20_abc_model_diagnostics.png'
DEF_PATH = FIG21_DIR / 'FigS21_def_model_diagnostics.png'
OVERLAP_MAP_PATH = FIG_DIR / 'FigS20_oof_overlap_map.png'
PERMUTATION_PATH = SOURCE_DIR / 'figs20_permutation_importance.csv'
ABLATION_PATH = SOURCE_DIR / 'figs20_ablation_metrics.csv'
FOLD_PATH = SOURCE21_DIR / 'figs21_spatial_fold_metrics.csv'
CALIBRATION_PATH = SOURCE21_DIR / 'figs21_calibration.csv'

EXPORT_DPI = 900
BLOCK_SIZE_KM = 50
N_SPATIAL_FOLDS = 5
PERMUTATION_REPEATS = 3
ANALYSIS_SEED = 20260712

main_doc = json.loads(MAIN_NOTEBOOK.read_text(encoding='utf-8'))
main_code_cells = [cell for cell in main_doc['cells'] if cell['cell_type'] == 'code']
main_namespace = globals()
for main_cell_index, main_cell in enumerate(main_code_cells[:3], start=1):
    main_source = ''.join(main_cell['source'])
    exec(compile(main_source, f'{MAIN_NOTEBOOK}:code-cell-{main_cell_index}', 'exec'), main_namespace)

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 8.5,
    'axes.labelsize': 8.5,
    'xtick.labelsize': 7.4,
    'ytick.labelsize': 7.4,
    'legend.fontsize': 7.2,
    'axes.linewidth': 0.7,
    'savefig.dpi': EXPORT_DPI,
    'savefig.bbox': 'tight',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

print('Mainline predictors:', len(FEATURE_COLUMNS))
print('Regression output:', REGRESSION_DIR)
print('Figure output:', FIG_DIR)



Cluster labels: outputs\RECON_MAIN_2011_2023\metrics\clustering\cluster_labels.csv
AEM raster: data\1 resistivity\aem_log10res_1km_masked.tif
Pumping table: data\train_val_test_inputs\GNN_spacetime\H1\aiwum_monthly.csv
Output: outputs\RECON_MAIN_2011_2023\metrics\regression


Coordinate check max error: 0.000000 m
AEM profile matrix: 87,871 cells x 70 layers
Native AEM depth range: 2.5-347.5 m
Valid response-labelled cells: 87,647


Predictor count: 77

AEM coverage by depth band among response-labelled cells:
    0- 15 m: 0.999
   15- 30 m: 0.999
   30- 50 m: 0.999
   50-100 m: 0.999
  100-150 m: 0.999
  150-200 m: 0.999
  200-300 m: 0.961
  300-400 m: 0.413

Response-class counts:
response_class
Fast recovery    33155
Slow recovery    30854
Buffered         23638
Mainline predictors: 77
Regression output: D:\Iowa paper\GNN to WTD\outputs\RECON_MAIN_2011_2023\metrics\regression
Figure output: D:\Iowa paper\GNN to WTD\outputs\RECON_MAIN_2011_2023\metrics\regression\figures


In [2]:
model_mask = valid_label_mask & np.any(
    np.isfinite(features[FINE_RESISTIVITY_FEATURES].to_numpy(dtype=float)),
    axis=1,
)
X = features.loc[model_mask, FEATURE_COLUMNS].to_numpy(dtype=float)
y = (features.loc[model_mask, 'response_class'] == TARGET_CLASS).to_numpy(dtype=int)
coords_xy = features.loc[model_mask, ['x', 'y']].to_numpy(dtype=float)

block_size_m = BLOCK_SIZE_KM * 1000.0
block_x = np.floor((coords_xy[:, 0] - coords_xy[:, 0].min()) / block_size_m).astype(int)
block_y = np.floor((coords_xy[:, 1] - coords_xy[:, 1].min()) / block_size_m).astype(int)
groups = block_x * 1000 + block_y

def make_model():
    return make_pipeline(
        SimpleImputer(strategy='median'),
        ExtraTreesClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=10,
            max_features=0.75,
            class_weight=None,
            random_state=42,
            n_jobs=-1,
        ),
    )

splitter = GroupKFold(n_splits=N_SPATIAL_FOLDS)
fold_splits = list(splitter.split(X, y, groups))
fold_models = []
oof_pred = np.full(len(y), np.nan, dtype=np.float32)
fold_rows = []

for fold, (train_idx, test_idx) in enumerate(fold_splits, start=1):
    model = make_model()
    model.fit(X[train_idx], y[train_idx])
    pred = model.predict_proba(X[test_idx])[:, 1]
    oof_pred[test_idx] = pred
    fold_models.append(model)
    fold_rows.append({
        'fold': fold,
        'n_test': len(test_idx),
        'target_prevalence': y[test_idx].mean(),
        'roc_auc': roc_auc_score(y[test_idx], pred),
        'average_precision': average_precision_score(y[test_idx], pred),
        'brier_score': brier_score_loss(y[test_idx], pred),
    })

if not np.isfinite(oof_pred).all():
    raise RuntimeError('Spatial OOF predictions contain missing values.')

fold_metrics = pd.DataFrame(fold_rows)
fold_metrics.to_csv(FOLD_PATH, index=False, lineterminator='\n')

equal_area_cutoff = float(np.quantile(oof_pred, 1.0 - y.mean()))

print(
    f'OOF ROC-AUC={roc_auc_score(y, oof_pred):.3f}; '
    f'AP={average_precision_score(y, oof_pred):.3f}; '
    f'Brier={brier_score_loss(y, oof_pred):.3f}'
)
print('Equal-area cutoff:', f'{equal_area_cutoff:.4f}')



OOF ROC-AUC=0.809; AP=0.694; Brier=0.167
Equal-area cutoff: 0.4684


In [3]:
native_layer_features = [
    name for name in NATIVE_PROFILE_FEATURES
    if name.startswith('logrho_native_')
]
native_summary_features = [
    name for name in NATIVE_PROFILE_FEATURES
    if name.startswith('native_logrho_')
]
pumping_features = list(LOG_PUMPING_FEATURES)

def depth_features(zmin, zmax):
    return [
        name for name, depth in zip(native_layer_features, native_depths_m)
        if zmin <= float(depth) < zmax
    ]

permutation_groups = {
    'AEM profile': native_layer_features,
    'AEM summaries': native_summary_features,
    'Pumping': pumping_features,
    'AEM 0-50 m': depth_features(0, 50),
    'AEM 50-150 m': depth_features(50, 150),
    'AEM 150-300 m': depth_features(150, 300),
    'AEM 300-350 m': depth_features(300, 350),
}
feature_to_index = {name: i for i, name in enumerate(FEATURE_COLUMNS)}
rng = np.random.default_rng(ANALYSIS_SEED)
permutation_rows = []

for fold, ((train_idx, test_idx), model) in enumerate(zip(fold_splits, fold_models), start=1):
    baseline = model.predict_proba(X[test_idx])[:, 1]
    baseline_auc = roc_auc_score(y[test_idx], baseline)
    baseline_ap = average_precision_score(y[test_idx], baseline)
    baseline_brier = brier_score_loss(y[test_idx], baseline)

    for group_name, group_features in permutation_groups.items():
        group_idx = np.asarray([feature_to_index[name] for name in group_features], dtype=int)
        for repeat in range(PERMUTATION_REPEATS):
            perm_order = rng.permutation(len(test_idx))
            X_permuted = X[test_idx].copy()
            X_permuted[:, group_idx] = X[test_idx][perm_order][:, group_idx]
            permuted = model.predict_proba(X_permuted)[:, 1]
            permutation_rows.append({
                'fold': fold,
                'repeat': repeat + 1,
                'group': group_name,
                'baseline_roc_auc': baseline_auc,
                'permuted_roc_auc': roc_auc_score(y[test_idx], permuted),
                'delta_roc_auc': baseline_auc - roc_auc_score(y[test_idx], permuted),
                'baseline_average_precision': baseline_ap,
                'permuted_average_precision': average_precision_score(y[test_idx], permuted),
                'delta_average_precision': baseline_ap - average_precision_score(y[test_idx], permuted),
                'baseline_brier': baseline_brier,
                'permuted_brier': brier_score_loss(y[test_idx], permuted),
                'delta_brier': brier_score_loss(y[test_idx], permuted) - baseline_brier,
            })

permutation = pd.DataFrame(permutation_rows)
permutation.to_csv(PERMUTATION_PATH, index=False, lineterminator='\n')

ablation_feature_sets = {
    'AEM structure only': NATIVE_PROFILE_FEATURES,
    'Pumping only': LOG_PUMPING_FEATURES,
    'AEM + pumping': FEATURE_COLUMNS,
}
ablation_rows = []
for model_name, model_features in ablation_feature_sets.items():
    feature_idx = np.asarray([feature_to_index[name] for name in model_features], dtype=int)
    predictions = np.full(len(y), np.nan, dtype=float)
    for train_idx, test_idx in fold_splits:
        model = make_model()
        model.fit(X[train_idx][:, feature_idx], y[train_idx])
        predictions[test_idx] = model.predict_proba(X[test_idx][:, feature_idx])[:, 1]
    ablation_rows.append({
        'model': model_name,
        'n_features': len(model_features),
        'roc_auc': roc_auc_score(y, predictions),
        'average_precision': average_precision_score(y, predictions),
        'brier_score': brier_score_loss(y, predictions),
    })

ablation = pd.DataFrame(ablation_rows)
ablation.to_csv(ABLATION_PATH, index=False, lineterminator='\n')

calibration_fraction, calibration_mean = calibration_curve(
    y, oof_pred, n_bins=10, strategy='quantile'
)
calibration = pd.DataFrame({
    'mean_predicted_probability': calibration_mean,
    'observed_fraction': calibration_fraction,
})
calibration.to_csv(CALIBRATION_PATH, index=False, lineterminator='\n')

print('Permutation groups:', ', '.join(permutation_groups))
print(ablation.to_string(index=False))
print('Saved source tables:', PERMUTATION_PATH, ABLATION_PATH, FOLD_PATH, CALIBRATION_PATH)



Permutation groups: AEM profile, AEM summaries, Pumping, AEM 0-50 m, AEM 50-150 m, AEM 150-300 m, AEM 300-350 m
             model  n_features  roc_auc  average_precision  brier_score
AEM structure only          73 0.787032           0.663283     0.175883
      Pumping only           4 0.578638           0.429715     0.226410
     AEM + pumping          77 0.808689           0.693865     0.166829
Saved source tables: D:\Iowa paper\GNN to WTD\outputs\figures\FigS20\source_data\figs20_permutation_importance.csv D:\Iowa paper\GNN to WTD\outputs\figures\FigS20\source_data\figs20_ablation_metrics.csv D:\Iowa paper\GNN to WTD\outputs\figures\FigS21\source_data\figs21_spatial_fold_metrics.csv D:\Iowa paper\GNN to WTD\outputs\figures\FigS21\source_data\figs21_calibration.csv


In [4]:
def style_axis(ax, grid_axis=None):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.7)
        spine.set_color('#333333')
    ax.tick_params(length=2.6, width=0.65, direction='out')
    if grid_axis:
        ax.grid(axis=grid_axis, color='#E1E1E1', linewidth=0.4, zorder=0)
        ax.set_axisbelow(True)

def panel_label(ax, label, fontsize=10):
    ax.text(-0.16, 1.04, label, transform=ax.transAxes,
            fontsize=fontsize, fontweight='bold', va='bottom', ha='left')

def summary_stats(frame, value_col):
    return frame.groupby('group')[value_col].agg(
        median='median',
        q25=lambda s: s.quantile(0.25),
        q75=lambda s: s.quantile(0.75),
    )

broad_groups = ['AEM profile', 'AEM summaries', 'Pumping']
broad_permutation = permutation[permutation.group.isin(broad_groups)].copy()
broad_permutation['positive_delta'] = broad_permutation['delta_roc_auc'].clip(lower=0)
broad_totals = broad_permutation.groupby(['fold', 'repeat'])['positive_delta'].transform('sum')
broad_permutation['share_percent'] = 100.0 * broad_permutation['positive_delta'] / broad_totals.replace(0, np.nan)
broad_stats = summary_stats(broad_permutation, 'share_percent').reindex(broad_groups)
fine_groups = ['AEM 0-50 m', 'AEM 50-150 m', 'AEM 150-300 m', 'AEM 300-350 m', 'AEM summaries', 'Pumping']
fine_stats = summary_stats(permutation[permutation.group.isin(fine_groups)], 'delta_roc_auc').reindex(fine_groups)

fig = plt.figure(figsize=(13.2, 8.6), dpi=EXPORT_DPI, facecolor='white')
grid = fig.add_gridspec(2, 4, left=0.055, right=0.985, bottom=0.065, top=0.96, wspace=0.42, hspace=0.48)
ax_a = fig.add_subplot(grid[0, 0])
ax_b = fig.add_subplot(grid[0, 1])
ax_c = fig.add_subplot(grid[0, 2])
panel_d = grid[0, 3].subgridspec(2, 1, hspace=0.38)
ax_d_roc = fig.add_subplot(panel_d[0, 0])
ax_d_pr = fig.add_subplot(panel_d[1, 0])
ax_e = fig.add_subplot(grid[1, 0])
ax_f = fig.add_subplot(grid[1, 1])
ax_g = fig.add_subplot(grid[1, 2:4])

x_a = np.arange(len(broad_groups))
ax_a.bar(
    x_a, broad_stats['median'],
    yerr=[broad_stats['median'] - broad_stats['q25'], broad_stats['q75'] - broad_stats['median']],
    color=['#4C78A8', '#9B8AC4', '#D78C45'], alpha=0.88, capsize=2.0,
    error_kw={'elinewidth': 0.65, 'ecolor': '#444444'},
)
ax_a.set_xticks(x_a)
ax_a.set_xticklabels(['AEM\nprofile', 'AEM\nsummaries', 'Pumping'])
ax_a.set_ylabel('Percentage (%)')
ax_a.set_ylim(0, 100)
ax_a.grid(axis='y', color='#E1E1E1', linewidth=0.4)
style_axis(ax_a)
panel_label(ax_a, 'a')

x_b = np.arange(len(fine_groups))
bar_colors = ['#3B73B9', '#5F91C1', '#88AFCB', '#B5C9D8', '#9B8AC4', '#D78C45']
ax_b.bar(
    x_b, fine_stats['median'], yerr=[
        fine_stats['median'] - fine_stats['q25'],
        fine_stats['q75'] - fine_stats['median'],
    ],
    color=bar_colors, alpha=0.88, capsize=2.0,
    error_kw={'elinewidth': 0.65, 'ecolor': '#444444'},
)
ax_b.axhline(0, color='#444444', linewidth=0.6)
ax_b.set_xticks(x_b)
ax_b.set_xticklabels(['0-50', '50-150', '150-300', '300-350', 'AEM\nsummary', 'Pumping'], rotation=35, ha='right')
ax_b.set_ylabel('ΔROC-AUC')
ax_b.grid(axis='y', color='#E1E1E1', linewidth=0.4)
style_axis(ax_b)
panel_label(ax_b, 'b')

metric_names = ['roc_auc', 'average_precision', 'brier_score']
metric_labels = ['ROC-AUC', 'AP', 'Brier']
ablation_matrix = ablation.set_index('model')[metric_names].reindex(
    ['AEM structure only', 'Pumping only', 'AEM + pumping']
).to_numpy(dtype=float)
ax_c.imshow(ablation_matrix, cmap='YlGnBu', vmin=0, vmax=1, aspect='auto')
ax_c.set_xticks(np.arange(3))
ax_c.set_xticklabels(metric_labels)
ax_c.set_yticks(np.arange(3))
ax_c.set_yticklabels(['AEM', 'Pumping', 'AEM + Pumping'])
for i in range(ablation_matrix.shape[0]):
    for j in range(ablation_matrix.shape[1]):
        ax_c.text(j, i, f'{ablation_matrix[i, j]:.3f}', ha='center', va='center', fontsize=8.0,
                  color='white' if ablation_matrix[i, j] > 0.55 else '#111111')
ax_c.set_title('Spatial-block ablation', loc='left', fontsize=9.5, fontweight='bold')
style_axis(ax_c)
panel_label(ax_c, 'c')

fpr, tpr, _ = roc_curve(y, oof_pred)
precision, recall, _ = precision_recall_curve(y, oof_pred)
ax_d_roc.plot(fpr, tpr, color='#2D5FB8', linewidth=1.7)
ax_d_roc.plot([0, 1], [0, 1], color='#999999', linestyle='--', linewidth=0.65)
ax_d_roc.set_xlabel('False-positive rate')
ax_d_roc.set_ylabel('True-positive rate')
ax_d_roc.set_title(f'ROC (AUC={roc_auc_score(y, oof_pred):.3f})', fontsize=8.0, loc='left')
style_axis(ax_d_roc)
ax_d_pr.plot(recall, precision, color='#C44E72', linewidth=1.7)
ax_d_pr.axhline(y.mean(), color='#999999', linestyle='--', linewidth=0.65)
ax_d_pr.set_xlabel('Recall')
ax_d_pr.set_ylabel('Precision')
ax_d_pr.set_title(f'PR (AP={average_precision_score(y, oof_pred):.3f})', fontsize=8.0, loc='left')
style_axis(ax_d_pr)
panel_label(ax_d_roc, 'd')

ax_e.plot([0, 1], [0, 1], color='#999999', linestyle='--', linewidth=0.65)
ax_e.plot(calibration['mean_predicted_probability'], calibration['observed_fraction'],
          marker='o', markersize=4.0, color='#3D7F55', linewidth=1.5)
ax_e.set_xlim(0, 1)
ax_e.set_ylim(0, 1)
ax_e.set_xlabel('Mean predicted probability')
ax_e.set_ylabel('Observed Slow-recovery fraction')
ax_e.set_title(f'Calibration (Brier={brier_score_loss(y, oof_pred):.3f})',
               loc='left', fontsize=8.8, fontweight='bold')
style_axis(ax_e)
panel_label(ax_e, 'e')

fold_plot = fold_metrics.set_index('fold')[['roc_auc', 'average_precision', 'brier_score']]
fold_x = np.arange(1, len(fold_plot) + 1)
for metric, color, label in [
    ('roc_auc', '#2D5FB8', 'ROC-AUC'),
    ('average_precision', '#C44E72', 'AP'),
    ('brier_score', '#D78C45', 'Brier'),
]:
    ax_f.plot(fold_x, fold_plot[metric], marker='o', markersize=3.5,
              linewidth=1.3, color=color, label=label)
ax_f.set_xticks(fold_x)
ax_f.set_xlabel('Spatial fold')
ax_f.set_ylabel('Metric value')
ax_f.set_ylim(0, 1)
ax_f.legend(frameon=False, loc='lower left', fontsize=6.8)
ax_f.grid(axis='y', color='#E1E1E1', linewidth=0.4)
style_axis(ax_f)
panel_label(ax_f, 'f')

probability_table = pd.read_csv(REGRESSION_DIR / 'best_extratrees_slow_recovery_probabilities.csv')
map_labels = probability_table.copy()
map_model_mask = map_labels['P_slow_recovery_extratrees_oof'].notna().to_numpy()
map_true = map_labels['is_slow_recovery_class'].astype(bool).to_numpy()
map_high = map_labels['high_P_slow_recovery_equal_area_oof'].astype(bool).to_numpy()

n_rows = int(map_labels['row'].max()) + 1
n_cols = int(map_labels['col'].max()) + 1
x_centres = map_labels.groupby('col')['x'].first().sort_index().to_numpy(dtype=float)
y_centres = map_labels.groupby('row')['y'].first().sort_index().to_numpy(dtype=float)
dx = float(np.nanmedian(np.diff(x_centres)))
dy = float(np.nanmedian(np.diff(y_centres)))
x_edges = np.r_[x_centres[0] - 0.5 * dx, x_centres + 0.5 * dx]
y_edges = np.r_[y_centres[0] - 0.5 * dy, y_centres + 0.5 * dy]
x_mesh, y_mesh = np.meshgrid(x_edges, y_edges)
lon_edges, lat_edges = Transformer.from_crs(
    'EPSG:5070', 'EPSG:4326', always_xy=True
).transform(x_mesh, y_mesh)

comparison = np.full(len(map_labels), np.nan, dtype=float)
comparison[map_model_mask] = 0
comparison[map_true & ~map_high & map_model_mask] = 1
comparison[~map_true & map_high & map_model_mask] = 2
comparison[map_true & map_high & map_model_mask] = 3
map_raster = np.full((n_rows, n_cols), np.nan, dtype=float)
map_raster[
    map_labels['row'].to_numpy(dtype=int),
    map_labels['col'].to_numpy(dtype=int),
] = comparison

boundary_lonlat = gpd.read_file(MRVA_BOUNDARY_PATH).to_crs('EPSG:4326')
boundary_geometry = boundary_lonlat.geometry.union_all()
river_segments = []
if MISSISSIPPI_RIVER_GMT_PATH.exists():
    current = []
    for line in MISSISSIPPI_RIVER_GMT_PATH.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith('>'):
            if current:
                river_segments.append(np.asarray(current, dtype=float))
                current = []
        else:
            lon, lat = line.split()[:2]
            current.append((float(lon), float(lat)))
    if current:
        river_segments.append(np.asarray(current, dtype=float))

comparison_cmap = ListedColormap(['#EFEFEF', '#557A9C', '#E4A65A', '#A43C4B'])
comparison_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], comparison_cmap.N)

def plot_overlap_map(ax, add_legend=True):
    ax.pcolormesh(
        lon_edges, lat_edges, map_raster,
        cmap=comparison_cmap, norm=comparison_norm,
        shading='flat', rasterized=True,
    )
    boundary_lonlat.boundary.plot(ax=ax, color='#1F2933', linewidth=0.55, zorder=4)
    for segment in river_segments:
        geometry = LineString(segment).intersection(boundary_geometry)
        if geometry.is_empty:
            continue
        if geometry.geom_type == 'LineString':
            xs, ys = geometry.xy
            ax.plot(xs, ys, color='#B7DDE8', linewidth=0.45, zorder=5)
        elif hasattr(geometry, 'geoms'):
            for part in geometry.geoms:
                xs, ys = part.xy
                ax.plot(xs, ys, color='#B7DDE8', linewidth=0.45, zorder=5)
    minx, miny, maxx, maxy = boundary_lonlat.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)
    ax.set_aspect(1.0 / np.cos(np.deg2rad((miny + maxy) / 2.0)))
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    if add_legend:
        handles = [
            Patch(facecolor='#EFEFEF', edgecolor='none', label='Neither'),
            Patch(facecolor='#557A9C', edgecolor='none', label='Slow only'),
            Patch(facecolor='#E4A65A', edgecolor='none', label='High P(Slow) only'),
            Patch(facecolor='#A43C4B', edgecolor='none', label='Both'),
        ]
        ax.legend(handles=handles, loc='lower right', frameon=False, fontsize=6.8)

plot_overlap_map(ax_g, add_legend=True)
panel_label(ax_g, 'g')

fig_map, ax_map = plt.subplots(figsize=(3.35, 5.8), dpi=EXPORT_DPI, facecolor='white')
plot_overlap_map(ax_map, add_legend=True)
fig_map.savefig(OVERLAP_MAP_PATH, dpi=EXPORT_DPI, facecolor='white')
plt.close(fig)
plt.close(fig_map)

print('Saved:', FIG_PATH)
print('Saved:', OVERLAP_MAP_PATH)
print('Permutation source:', PERMUTATION_PATH)



Saved: D:\Iowa paper\GNN to WTD\outputs\figures\FigS20\FigS20_abc_model_diagnostics.png
Saved: D:\Iowa paper\GNN to WTD\outputs\figures\FigS20\FigS20_oof_overlap_map.png
Permutation source: D:\Iowa paper\GNN to WTD\outputs\figures\FigS20\source_data\figs20_permutation_importance.csv


In [5]:
legacy_path = FIG_PATH.parent / 'FigS20_model_diagnostics.png'
legacy_path.unlink(missing_ok=True)

fig_abc, (ax_a, ax_b, ax_c) = plt.subplots(1, 3, figsize=(13.2, 4.55), dpi=EXPORT_DPI, facecolor='white')
fig_abc.subplots_adjust(left=0.058, right=0.985, bottom=0.20, top=0.92, wspace=0.42)

x_a = np.arange(len(broad_groups))
ax_a.bar(
    x_a, broad_stats['median'],
    yerr=[broad_stats['median'] - broad_stats['q25'], broad_stats['q75'] - broad_stats['median']],
    color=['#4C78A8', '#9B8AC4', '#D78C45'], alpha=0.88, capsize=2.0,
    error_kw={'elinewidth': 0.65, 'ecolor': '#444444'},
)
ax_a.set_xticks(x_a)
ax_a.set_xticklabels(['AEM\nprofile', 'AEM\nsummaries', 'Pumping'], fontsize=11.0, fontweight='medium')
ax_a.set_ylabel('Percentage (%)', fontsize=12.0, labelpad=7)
ax_a.set_ylim(0, 100)
ax_a.grid(axis='y', color='#E1E1E1', linewidth=0.4)
style_axis(ax_a)
panel_label(ax_a, 'a', fontsize=13.0)

x_b = np.arange(len(fine_groups))
bar_colors = ['#3B73B9', '#5F91C1', '#88AFCB', '#B5C9D8', '#9B8AC4', '#D78C45']
ax_b.bar(
    x_b, fine_stats['median'],
    yerr=[fine_stats['median'] - fine_stats['q25'], fine_stats['q75'] - fine_stats['median']],
    color=bar_colors, alpha=0.88, capsize=2.0,
    error_kw={'elinewidth': 0.65, 'ecolor': '#444444'},
)
ax_b.axhline(0, color='#444444', linewidth=0.6)
ax_b.set_xticks(x_b)
ax_b.set_xticklabels(['0-50', '50-150', '150-300', '300-350', 'AEM\nsummary', 'Pumping'], rotation=35, ha='right')
ax_b.set_ylabel(r'$\Delta$ROC-AUC')
ax_b.grid(axis='y', color='#E1E1E1', linewidth=0.4)
style_axis(ax_b)
panel_label(ax_b, 'b', fontsize=11.0)

metric_names = ['roc_auc', 'average_precision', 'brier_score']
metric_labels = ['ROC-AUC', 'AP', 'Brier']
ablation_matrix = ablation.set_index('model')[metric_names].reindex(
    ['AEM structure only', 'Pumping only', 'AEM + pumping']
).to_numpy(dtype=float)
ax_c.imshow(ablation_matrix, cmap='YlGnBu', vmin=0, vmax=1, aspect='auto')
ax_c.set_xticks(np.arange(3))
ax_c.set_xticklabels(metric_labels)
ax_c.set_yticks(np.arange(3))
ax_c.set_yticklabels(['AEM', 'Pumping', 'AEM + Pumping'])
for i in range(ablation_matrix.shape[0]):
    for j in range(ablation_matrix.shape[1]):
        ax_c.text(j, i, f'{ablation_matrix[i, j]:.3f}', ha='center', va='center', fontsize=8.5,
                  color='white' if ablation_matrix[i, j] > 0.55 else '#111111')
ax_c.set_title('Spatial-block ablation', loc='left', fontsize=10.0, fontweight='bold')
style_axis(ax_c)
panel_label(ax_c, 'c', fontsize=11.0)
for axis in (ax_a, ax_b, ax_c):
    axis.xaxis.label.set_size(10.0)
    axis.yaxis.label.set_size(10.0)
    axis.tick_params(axis='both', labelsize=8.5)
ax_a.tick_params(axis='x', labelsize=11.0, pad=3)
ax_a.tick_params(axis='y', labelsize=10.5, pad=3)
fig_abc.savefig(FIG_PATH, dpi=EXPORT_DPI, facecolor='white', bbox_inches='tight')
plt.close(fig_abc)

fig_def, axes_def = plt.subplots(2, 2, figsize=(9.5, 8.0), dpi=EXPORT_DPI, facecolor='white')
fig_def.subplots_adjust(left=0.10, right=0.97, bottom=0.15, top=0.94, wspace=0.34, hspace=0.42)
ax_d_roc, ax_e = axes_def[0]
ax_d_pr, ax_f = axes_def[1]
fpr, tpr, _ = roc_curve(y, oof_pred)
precision, recall, _ = precision_recall_curve(y, oof_pred)
ax_d_roc.plot(fpr, tpr, color='#2D5FB8', linewidth=1.7)
ax_d_roc.plot([0, 1], [0, 1], color='#999999', linestyle='--', linewidth=0.65)
ax_d_roc.set_xlabel('False-positive rate')
ax_d_roc.set_ylabel('True-positive rate')
ax_d_roc.set_title(f'ROC (AUC={roc_auc_score(y, oof_pred):.3f})', fontsize=8.8, loc='left', fontweight='bold')
style_axis(ax_d_roc)
panel_label(ax_d_roc, 'd')
ax_d_pr.plot(recall, precision, color='#C44E72', linewidth=1.7)
ax_d_pr.axhline(y.mean(), color='#999999', linestyle='--', linewidth=0.65)
ax_d_pr.set_xlabel('Recall')
ax_d_pr.set_ylabel('Precision')
ax_d_pr.set_title(f'Precision-recall (AP={average_precision_score(y, oof_pred):.3f})', fontsize=8.8, loc='left', fontweight='bold')
style_axis(ax_d_pr)

ax_e.plot([0, 1], [0, 1], color='#999999', linestyle='--', linewidth=0.65)
ax_e.plot(calibration['mean_predicted_probability'], calibration['observed_fraction'],
          marker='o', markersize=4.0, color='#3D7F55', linewidth=1.5)
ax_e.set_xlim(0, 1)
ax_e.set_ylim(0, 1)
ax_e.set_xlabel('Mean predicted probability')
ax_e.set_ylabel('Observed Slow-recovery fraction')
ax_e.set_title(f'Calibration (Brier={brier_score_loss(y, oof_pred):.3f})',
               loc='left', fontsize=8.8, fontweight='bold')
style_axis(ax_e)
panel_label(ax_e, 'e')

fold_plot = fold_metrics.set_index('fold')[['roc_auc', 'average_precision', 'brier_score']]
fold_x = np.arange(1, len(fold_plot) + 1)
for metric, color, label in [
    ('roc_auc', '#2D5FB8', 'ROC-AUC'),
    ('average_precision', '#C44E72', 'AP'),
    ('brier_score', '#D78C45', 'Brier'),
]:
    ax_f.plot(fold_x, fold_plot[metric], marker='o', markersize=3.5,
              linewidth=1.3, color=color, label=label)
ax_f.set_xticks(fold_x)
ax_f.set_xlabel('Spatial fold')
ax_f.set_ylabel('Metric value')
ax_f.set_ylim(0, 1)
ax_f.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=3, fontsize=6.8, borderaxespad=0)
ax_f.grid(axis='y', color='#E1E1E1', linewidth=0.4)
style_axis(ax_f)
panel_label(ax_f, 'f')
fig_def.savefig(DEF_PATH, dpi=EXPORT_DPI, facecolor='white', bbox_inches='tight')
plt.close(fig_def)

fig_map, ax_map = plt.subplots(figsize=(3.35, 5.8), dpi=EXPORT_DPI, facecolor='white')
plot_overlap_map(ax_map, add_legend=True)
panel_label(ax_map, 'g')
fig_map.savefig(OVERLAP_MAP_PATH, dpi=EXPORT_DPI, facecolor='white', bbox_inches='tight')
plt.close(fig_map)

print('Saved:', FIG_PATH)
print('Saved:', DEF_PATH)
print('Saved:', OVERLAP_MAP_PATH)


Saved: D:\Iowa paper\GNN to WTD\outputs\figures\FigS20\FigS20_abc_model_diagnostics.png
Saved: D:\Iowa paper\GNN to WTD\outputs\figures\FigS21\FigS21_def_model_diagnostics.png
Saved: D:\Iowa paper\GNN to WTD\outputs\figures\FigS20\FigS20_oof_overlap_map.png
